# 

In [1]:
import sqlite3
import pandas as pd
import os

In [2]:
conn = sqlite3.connect("../data/database/career.db")
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()

onet_path = "../data/raw/onet"

tính toán cột normalized_weight

In [3]:
skills = pd.read_sql_query("""
    SELECT *
    FROM occupation_skills
""", conn)

print("Rows:", len(skills))
print("Unique occupations:", skills["occupation_code"].nunique())
print("Unique skills:", skills["skill_id"].nunique())

skills.head()

Rows: 9100
Unique occupations: 910
Unique skills: 10


,occupation_code,skill_id,skill_name,importance_score,level_score,normalized_weight
0,11-1011.00,2.A.1.a,Reading Comprehension,4.12,4.62,None
1,11-1011.00,2.A.1.b,Active Listening,4.00,4.75,None
2,11-1011.00,2.A.1.c,Writing,4.12,4.38,None
3,11-1011.00,2.A.1.d,Speaking,4.25,4.75,None
4,11-1011.00,2.A.1.e,Mathematics,3.25,3.50,None


In [4]:
print("Missing values:")
print(
    skills[
        ["importance_score", "level_score"]
    ].isna().sum()
)

Missing values:
importance_score    0
level_score         0
dtype: int64


In [5]:
print("Importance range:")
print(
    skills["importance_score"].min(),
    "→",
    skills["importance_score"].max()
)

print("Level range:")
print(
    skills["level_score"].min(),
    "→",
    skills["level_score"].max()
)

Importance range:
1.0 → 5.0
Level range:
0.0 → 6.0


In [6]:
skills["raw_weight"] = (
    skills["importance_score"]
    * skills["level_score"]
)
skills[
    [
        "occupation_code",
        "skill_id",
        "skill_name",
        "importance_score",
        "level_score",
        "raw_weight"
    ]
].head(10)

,occupation_code,skill_id,skill_name,importance_score,level_score,raw_weight
0,11-1011.00,2.A.1.a,Reading Comprehension,4.12,4.62,19.0344
1,11-1011.00,2.A.1.b,Active Listening,4.00,4.75,19.0000
2,11-1011.00,2.A.1.c,Writing,4.12,4.38,18.0456
3,11-1011.00,2.A.1.d,Speaking,4.25,4.75,20.1875
4,11-1011.00,2.A.1.e,Mathematics,3.25,3.50,11.3750
5,11-1011.00,2.A.1.f,Science,1.62,0.75,1.2150
6,11-1011.00,2.A.2.a,Critical Thinking,4.38,4.75,20.8050
7,11-1011.00,2.A.2.b,Active Learning,3.75,4.50,16.8750
8,11-1011.00,2.A.2.c,Learning Strategies,3.12,3.75,11.7000
9,11-1011.00,2.A.2.d,Monitoring,4.00,5.25,21.0000


In [7]:
skills["normalized_weight"] = (
    skills["raw_weight"]
    / skills.groupby("occupation_code")["raw_weight"]
      .transform("sum")
)

In [8]:
print(
    skills["normalized_weight"].describe()
)

count    9100.000000
mean        0.100000
std         0.037243
min         0.000000
25%         0.081598
50%         0.107474
75%         0.124964
max         0.231754
Name: normalized_weight, dtype: float64


In [9]:
# thử với 1 occupation
skills[
    skills["occupation_code"] == "11-1011.00"
][
    [
        "skill_name",
        "importance_score",
        "level_score",
        "raw_weight",
        "normalized_weight"
    ]
].sort_values(
    "normalized_weight",
    ascending=False
).head(10)

,skill_name,importance_score,level_score,raw_weight,normalized_weight
9,Monitoring,4.00,5.25,21.0000,0.131878
6,Critical Thinking,4.38,4.75,20.8050,0.130654
3,Speaking,4.25,4.75,20.1875,0.126776
0,Reading Comprehension,4.12,4.62,19.0344,0.119535
1,Active Listening,4.00,4.75,19.0000,0.119319
2,Writing,4.12,4.38,18.0456,0.113325
7,Active Learning,3.75,4.50,16.8750,0.105974
8,Learning Strategies,3.12,3.75,11.7000,0.073475
4,Mathematics,3.25,3.50,11.3750,0.071434
5,Science,1.62,0.75,1.2150,0.007630


In [10]:
# insert vào database
for _, row in skills.iterrows():
    conn.execute("""
        UPDATE occupation_skills
        SET normalized_weight = ?
        WHERE occupation_code = ?
          AND skill_id = ?
    """, (
        row["normalized_weight"],
        row["occupation_code"],
        row["skill_id"]
    ))

conn.commit()

print("Đã cập nhật normalized_weight.")

Đã cập nhật normalized_weight.


In [11]:
check = pd.read_sql_query("""
    SELECT
        occupation_code,
        skill_id,
        skill_name,
        importance_score,
        level_score,
        normalized_weight
    FROM occupation_skills
    LIMIT 10
""", conn)

check

,occupation_code,skill_id,skill_name,importance_score,level_score,normalized_weight
0,11-1011.00,2.A.1.a,Reading Comprehension,4.12,4.62,0.119535
1,11-1011.00,2.A.1.b,Active Listening,4.00,4.75,0.119319
2,11-1011.00,2.A.1.c,Writing,4.12,4.38,0.113325
3,11-1011.00,2.A.1.d,Speaking,4.25,4.75,0.126776
4,11-1011.00,2.A.1.e,Mathematics,3.25,3.50,0.071434
5,11-1011.00,2.A.1.f,Science,1.62,0.75,0.007630
6,11-1011.00,2.A.2.a,Critical Thinking,4.38,4.75,0.130654
7,11-1011.00,2.A.2.b,Active Learning,3.75,4.50,0.105974
8,11-1011.00,2.A.2.c,Learning Strategies,3.12,3.75,0.073475
9,11-1011.00,2.A.2.d,Monitoring,4.00,5.25,0.131878


Import chỉ số AI exposure từ OECD

In [3]:
oecd_raw = pd.read_excel(
    "../data/raw/oecd/OECD AI Capability Gap Index_public data.xlsx",
    sheet_name="Data"
)

print("Shape:", oecd_raw.shape)

print("\nColumns:")
print(oecd_raw.columns.tolist())

print("\nFirst 5 rows:")
display(oecd_raw.head())

Shape: (882, 22)

Columns:
['OCCUPATION IDENTIFIERS', 'Unnamed: 1', 'AI CAPABILITY GAP INDEX', 'Unnamed: 3', 'CAPABILITY GAP BY DOMAIN', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'CAPABILITY DEMANDS BY DOMAIN', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21']

First 5 rows:


,OCCUPATION IDENTIFIERS,Unnamed: 1,AI CAPABILITY GAP INDEX,Unnamed: 3,CAPABILITY GAP BY DOMAIN,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 12,CAPABILITY DEMANDS BY DOMAIN,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21
0,Occupation Code (O*NET-SOC Code),Occupation Title,"AI Capability Gap Index (Reversed, norm.)",AI Capability Gap Index (Total),Language — Capability Gap,Social Interaction — Capability Gap,Problem Solving — Capability Gap,Creativity — Capability Gap,Metacognition & Critical Thinking — Capability...,"Knowledge, Learning & Memory — Capability Gap",...,Robotic Intelligence — Capability Gap,Language — Job Demands,Social Interaction — Job Demands,Problem Solving — Job Demands,Creativity — Job Demands,Metacognition & Critical Thinking — Job Demands,"Knowledge, Learning & Memory — Job Demands",Vision — Job Demands,Manipulation — Job Demands,Robotic Intelligence — Job Demands
1,OCC_Code,OCC_Title,AI Capability Gap Index_Rev. norm.,AI Capability Gap Index_Total,Lang_Gap,Soc_Gap,ProbSolv_Gap,Creativity_Gap,Metacognition_Gap,KLM_Gap,...,Robot_Gap,Lang_Dmd,Soc_Dmd,ProbSolv_Dmd,Creativity_Dmd,Metacognition_Dmd,KLM_Dmd,Vision_Dmd,Manip_Dmd,Robot_Dmd
2,—,—,Numeric 0-1 (higher = more exposed),Numeric >0 (lower = more exposed),0–2 (lower = more exposed),0–3 (lower = more exposed),0–3 (lower = more exposed),0–2 (lower = more exposed),0–3 (lower = more exposed),0–2 (lower = more exposed),...,0–3 (lower = more exposed),0–5 scale,0–5 scale,0–5 scale,0–5 scale,0–5 scale,0–5 scale,0–5 scale,0–5 scale,0–5 scale
3,11-1011.00,Chief Executives,0.491063,11.705547,1.7,2.75,2.95,0.75,2.95,2,...,0,4.7,4.75,4.95,3.75,4.95,5,0.05,0,0
4,11-1011.03,Chief Sustainability Officers,0.696279,6.985579,1.05,2,2.25,0.2,2.25,1.25,...,0,4.05,4,4.25,3.2,4.25,4.25,0,0,0


In [4]:
print("Shape:", oecd_raw.shape)

display(oecd_raw.iloc[:6, :10])

Shape: (882, 22)


,OCCUPATION IDENTIFIERS,Unnamed: 1,AI CAPABILITY GAP INDEX,Unnamed: 3,CAPABILITY GAP BY DOMAIN,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,Occupation Code (O*NET-SOC Code),Occupation Title,"AI Capability Gap Index (Reversed, norm.)",AI Capability Gap Index (Total),Language — Capability Gap,Social Interaction — Capability Gap,Problem Solving — Capability Gap,Creativity — Capability Gap,Metacognition & Critical Thinking — Capability...,"Knowledge, Learning & Memory — Capability Gap"
1,OCC_Code,OCC_Title,AI Capability Gap Index_Rev. norm.,AI Capability Gap Index_Total,Lang_Gap,Soc_Gap,ProbSolv_Gap,Creativity_Gap,Metacognition_Gap,KLM_Gap
2,—,—,Numeric 0-1 (higher = more exposed),Numeric >0 (lower = more exposed),0–2 (lower = more exposed),0–3 (lower = more exposed),0–3 (lower = more exposed),0–2 (lower = more exposed),0–3 (lower = more exposed),0–2 (lower = more exposed)
3,11-1011.00,Chief Executives,0.491063,11.705547,1.7,2.75,2.95,0.75,2.95,2
4,11-1011.03,Chief Sustainability Officers,0.696279,6.985579,1.05,2,2.25,0.2,2.25,1.25
5,11-1021.00,General and Operations Managers,0.739312,5.995822,1,2,2.25,0,2.25,1


In [ ]:
# Lấy từ dòng thứ 3
oecd = oecd_raw.iloc[3:].copy()

oecd.columns = oecd_raw.iloc[1].astype(str).str.strip() # đổi dòng thứ 2 sang str và làm column

oecd = oecd.reset_index(drop=True)

print("Shape:", oecd.shape)
print(oecd.columns.tolist())

display(oecd.head())

Shape: (879, 22)
['OCC_Code', 'OCC_Title', 'AI Capability Gap Index_Rev. norm.', 'AI Capability Gap Index_Total', 'Lang_Gap', 'Soc_Gap', 'ProbSolv_Gap', 'Creativity_Gap', 'Metacognition_Gap', 'KLM_Gap', 'Vision_Gap', 'Manip_Gap', 'Robot_Gap', 'Lang_Dmd', 'Soc_Dmd', 'ProbSolv_Dmd', 'Creativity_Dmd', 'Metacognition_Dmd', 'KLM_Dmd', 'Vision_Dmd', 'Manip_Dmd', 'Robot_Dmd']


1,OCC_Code,OCC_Title,AI Capability Gap Index_Rev. norm.,AI Capability Gap Index_Total,Lang_Gap,Soc_Gap,ProbSolv_Gap,Creativity_Gap,Metacognition_Gap,KLM_Gap,...,Robot_Gap,Lang_Dmd,Soc_Dmd,ProbSolv_Dmd,Creativity_Dmd,Metacognition_Dmd,KLM_Dmd,Vision_Dmd,Manip_Dmd,Robot_Dmd
0,11-1011.00,Chief Executives,0.491063,11.705547,1.7,2.75,2.95,0.75,2.95,2,...,0,4.7,4.75,4.95,3.75,4.95,5,0.05,0,0
1,11-1011.03,Chief Sustainability Officers,0.696279,6.985579,1.05,2,2.25,0.2,2.25,1.25,...,0,4.05,4,4.25,3.2,4.25,4.25,0,0,0
2,11-1021.00,General and Operations Managers,0.739312,5.995822,1,2,2.25,0,2.25,1,...,0,4,4,4.25,2.95,4.25,4,1.45,1.05,1.6
3,11-2011.00,Advertising and Promotions Managers,0.738163,6.02226,1.15,2,2,0.55,2.25,0.8,...,0,4.15,4,4,3.55,4.25,3.8,0.15,0,0
4,11-2021.00,Marketing Managers,0.742922,5.912801,1,2,2.15,0.05,2.25,0.8,...,0,4,4,4.15,3.05,4.25,3.8,0.15,0,0


In [7]:
# Lấy 2 cột cần thiết
oecd_exposure = oecd[
    [
        "OCC_Code",
        "AI Capability Gap Index_Rev. norm."
    ]
].copy()

oecd_exposure = oecd_exposure.rename(columns={
    "OCC_Code": "occupation_code",
    "AI Capability Gap Index_Rev. norm.": "ai_exposure_score"
})

# Ép kiểu dữ liệu
oecd_exposure["occupation_code"] = (
    oecd_exposure["occupation_code"]
    .astype(str)
    .str.strip()
)

oecd_exposure["ai_exposure_score"] = pd.to_numeric(
    oecd_exposure["ai_exposure_score"],
    errors="coerce"# Nếu không ép kiểu số được gán thành NaN
)

In [8]:
print("Rows:", len(oecd_exposure))
print(
    "Unique occupations:",
    oecd_exposure["occupation_code"].nunique()
)

print("\nMissing:")
print(oecd_exposure.isna().sum())

print("\nExposure range:")
print(
    oecd_exposure["ai_exposure_score"].min(),
    "→",
    oecd_exposure["ai_exposure_score"].max()
)

Rows: 879
Unique occupations: 879

Missing:
1
occupation_code      0
ai_exposure_score    0
dtype: int64

Exposure range:
0.49106317256559395 → 1.0


In [9]:
duplicates = oecd_exposure.duplicated(
    "occupation_code"
).sum()

print("Duplicate occupation codes:", duplicates)

Duplicate occupation codes: 0


In [10]:
# Kiểm tra match với O*NET
onet_codes = set(
    pd.read_sql_query(
        "SELECT occupation_code FROM occupations",
        conn
    )["occupation_code"]
)

oecd_codes = set(
    oecd_exposure["occupation_code"]
)

matched = onet_codes & oecd_codes
oecd_unmatched = oecd_codes - onet_codes
onet_missing_exposure = onet_codes - oecd_codes

print("O*NET occupations:", len(onet_codes))
print("OECD occupations:", len(oecd_codes))

print("Matched:", len(matched))
print("OECD unmatched:", len(oecd_unmatched))
print("O*NET without OECD exposure:", len(onet_missing_exposure))

O*NET occupations: 1016
OECD occupations: 879
Matched: 879
OECD unmatched: 0
O*NET without OECD exposure: 137


In [11]:
exposure_check = pd.read_sql_query("""
    SELECT
        COUNT(*) AS total_occupations,
        SUM(
            CASE
                WHEN ai_exposure_score IS NOT NULL
                THEN 1
                ELSE 0
            END
        ) AS with_exposure,
        SUM(
            CASE
                WHEN ai_exposure_score IS NULL
                THEN 1
                ELSE 0
            END
        ) AS without_exposure
    FROM occupations
""", conn)

exposure_check

,total_occupations,with_exposure,without_exposure
0,1016,0,1016


In [12]:
for _, row in oecd_exposure.iterrows():
    conn.execute("""
        UPDATE occupations
        SET ai_exposure_score = ?
        WHERE occupation_code = ?
    """, (
        row["ai_exposure_score"],
        row["occupation_code"]
    ))

conn.commit()

print("Đã cập nhật ai_exposure_score.")

Đã cập nhật ai_exposure_score.


In [13]:
exposure_check = pd.read_sql_query("""
    SELECT
        COUNT(*) AS total_occupations,
        SUM(
            CASE
                WHEN ai_exposure_score IS NOT NULL
                THEN 1 ELSE 0
            END
        ) AS with_exposure,
        SUM(
            CASE
                WHEN ai_exposure_score IS NULL
                THEN 1 ELSE 0
            END
        ) AS without_exposure
    FROM occupations
""", conn)

exposure_check

,total_occupations,with_exposure,without_exposure
0,1016,879,137


## Insert vào bảng occupation_feature

In [14]:
# ============================================================
# CELL 1 — Đọc Work Activities
# ============================================================

work_raw = pd.read_csv(
    f"{onet_path}/work_activities.csv"
)

print("Shape:", work_raw.shape)

print("\nColumns:")
print(work_raw.columns.tolist())

display(work_raw.head())

Shape: (74702, 15)

Columns:
['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']


,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,4.A.1.a.1,Getting Information,IM,Importance,4.56,29.0,0.1559,4.2369,4.8756,N,NaN,08/2023,Incumbent
1,11-1011.00,Chief Executives,4.A.1.a.1,Getting Information,LV,Level,4.89,30.0,0.1727,4.5393,5.2458,N,N,08/2023,Incumbent
2,11-1011.00,Chief Executives,4.A.1.a.2,"Monitoring Processes, Materials, or Surroundings",IM,Importance,4.25,30.0,0.2125,3.8130,4.6823,N,NaN,08/2023,Incumbent
3,11-1011.00,Chief Executives,4.A.1.a.2,"Monitoring Processes, Materials, or Surroundings",LV,Level,5.21,30.0,0.3872,4.4133,5.9971,N,N,08/2023,Incumbent
4,11-1011.00,Chief Executives,4.A.1.b.1,"Identifying Objects, Actions, and Events",IM,Importance,4.23,29.0,0.1544,3.9180,4.5507,N,NaN,08/2023,Incumbent


O*NET có cả IM và LV. Ở đây ta dùng LV để biểu diễn mức độ của Work Activity đối với occupation.

In [16]:
# Chọn những cột cần thiết
work = work_raw[
    [
        "O*NET-SOC Code",
        "Element Name",
        "Scale ID",
        "Data Value"
    ]
].copy()

work = work.rename(columns={
    "O*NET-SOC Code": "occupation_code",
    "Element Name": "activity",
    "Scale ID": "scale",
    "Data Value": "score"
})

work["score"] = pd.to_numeric(
    work["score"],
    errors="coerce"
)

work.head()

,occupation_code,activity,scale,score
0,11-1011.00,Getting Information,IM,4.56
1,11-1011.00,Getting Information,LV,4.89
2,11-1011.00,"Monitoring Processes, Materials, or Surroundings",IM,4.25
3,11-1011.00,"Monitoring Processes, Materials, or Surroundings",LV,5.21
4,11-1011.00,"Identifying Objects, Actions, and Events",IM,4.23


In [17]:
print(work["scale"].value_counts())

print("\nMissing scores:")
print(work["score"].isna().sum())

print("\nScore range:")
print(
    work.groupby("scale")["score"]
    .agg(["count", "min", "max", "mean", "std"])
)

scale
IM    37351
LV    37351
Name: count, dtype: int64

Missing scores:
0

Score range:
       count  min   max      mean       std
scale                                      
IM     37351  1.0  5.00  3.167122  0.866014
LV     37351  0.0  6.94  3.355221  1.331980


Cell 5 — Pivot IM và LV thành hai cột

In [18]:
work = work.pivot_table(
    index=["occupation_code", "activity"],
    columns="scale",
    values="score",
    aggfunc="first"
).reset_index()

work.columns.name = None

work = work.rename(columns={
    "IM": "importance",
    "LV": "level"
})

work.head()

,occupation_code,activity,importance,level
0,11-1011.00,Analyzing Data or Information,4.38,4.84
1,11-1011.00,Assisting and Caring for Others,3.31,3.10
2,11-1011.00,Coaching and Developing Others,4.71,5.85
3,11-1011.00,Communicating with People Outside the Organiza...,4.55,5.47
4,11-1011.00,"Communicating with Supervisors, Peers, or Subo...",4.75,5.82


In [19]:
# Kiểm tra missing value
print("Missing importance:", work["importance"].isna().sum())
print("Missing level:", work["level"].isna().sum())

Missing importance: 0
Missing level: 0


In [34]:
activity_lv = (
    work
    .groupby("activity")["level"]
    .mean()
    .sort_values(ascending=False)
)

display(activity_lv)

activity
Updating and Using Relevant Knowledge                                              4.592964
Organizing, Planning, and Prioritizing Work                                        4.522053
Establishing and Maintaining Interpersonal Relationships                           4.509396
Making Decisions and Solving Problems                                              4.437069
Communicating with Supervisors, Peers, or Subordinates                             4.409111
Getting Information                                                                4.215258
Identifying Objects, Actions, and Events                                           4.164962
Monitoring Processes, Materials, or Surroundings                                   4.128046
Processing Information                                                             4.119243
Thinking Creatively                                                                3.947135
Analyzing Data or Information                                          

In [35]:
activity_im = (
    work
    .groupby("activity")["importance"]
    .mean()
    .sort_values(ascending=False)
)

display(activity_im)

activity
Getting Information                                                                4.232854
Communicating with Supervisors, Peers, or Subordinates                             4.026465
Making Decisions and Solving Problems                                              4.019528
Identifying Objects, Actions, and Events                                           3.858397
Updating and Using Relevant Knowledge                                              3.805488
Organizing, Planning, and Prioritizing Work                                        3.727947
Establishing and Maintaining Interpersonal Relationships                           3.687629
Documenting/Recording Information                                                  3.676597
Processing Information                                                             3.663326
Working with Computers                                                             3.644775
Monitoring Processes, Materials, or Surroundings                       

Do corr không lí tưởng để dùng k-mean nên tinh chỉnh lại FEATURE_GROUPS

In [36]:
FEATURE_GROUPS = {

    "cognitive_intensity": [
        "Making Decisions and Solving Problems",
        "Thinking Creatively",
        "Developing Objectives and Strategies",
        "Updating and Using Relevant Knowledge",
        "Judging the Qualities of Objects, Services, or People",
        "Evaluating Information to Determine Compliance with Standards",
        "Estimating the Quantifiable Characteristics of Products, Events, or Information",
        "Organizing, Planning, and Prioritizing Work",
    ],

    "social_intensity": [
        "Communicating with Supervisors, Peers, or Subordinates",
        "Communicating with People Outside the Organization",
        "Establishing and Maintaining Interpersonal Relationships",
        "Assisting and Caring for Others",
        "Selling or Influencing Others",
        "Resolving Conflicts and Negotiating with Others",
        "Performing for or Working Directly with the Public",
        "Coordinating the Work and Activities of Others",
        "Developing and Building Teams",
        "Training and Teaching Others",
        "Guiding, Directing, and Motivating Subordinates",
        "Coaching and Developing Others",
        "Providing Consultation and Advice to Others",
        "Interpreting the Meaning of Information for Others",
        "Staffing Organizational Units",
    ],

    "physical_operational_intensity": [
        "Performing General Physical Activities",
        "Handling and Moving Objects",
        "Operating Vehicles, Mechanized Devices, or Equipment",
        "Repairing and Maintaining Mechanical Equipment",
        "Repairing and Maintaining Electronic Equipment",
        "Controlling Machines and Processes",
        "Inspecting Equipment, Structures, or Materials",
    ],

    "information_digital_intensity": [
        "Working with Computers",
        "Processing Information",
        "Analyzing Data or Information",
        "Getting Information",
        "Drafting, Laying Out, and Specifying Technical Devices, Parts, and Equipment",
    ],

    "administrative_routine_intensity": [
        "Monitoring Processes, Materials, or Surroundings",
        "Scheduling Work and Activities",
        "Documenting/Recording Information",
        "Performing Administrative Activities",
        "Monitoring and Controlling Resources",
        "Identifying Objects, Actions, and Events",
    ],
}

In [37]:
occupation_df = pd.read_sql_query(
    """
    SELECT occupation_code
    FROM occupations
    """,
    conn
)

features = occupation_df.copy()

for feature_name, activities in FEATURE_GROUPS.items():

    score = (
        work[
            work["activity"].isin(activities)
        ]
        .groupby("occupation_code")["level"]
        .mean()
        .rename(feature_name)
    )

    features = features.merge(
        score,
        on="occupation_code",
        how="left"
    )

features.head()

,occupation_code,cognitive_intensity,social_intensity,physical_operational_intensity,information_digital_intensity,administrative_routine_intensity
0,11-1011.00,4.88125,5.149333,1.524286,3.798,4.658333
1,11-1011.03,5.09500,4.902000,1.308571,4.058,4.275000
2,11-1021.00,3.93250,4.120000,2.061429,3.648,3.818333
3,11-1031.00,NaN,NaN,NaN,NaN,NaN
4,11-2011.00,4.16500,3.882000,1.381429,2.878,3.950000


In [38]:
all_activities = set(work["activity"].dropna())

for feature_name, activities in FEATURE_GROUPS.items():
    missing = [
        activity
        for activity in activities
        if activity not in all_activities
    ]

    print(f"\n{feature_name}")

    if missing:
        print("Missing activities:")
        for activity in missing:
            print(" -", activity)
    else:
        print("All activities found.")


cognitive_intensity
All activities found.

social_intensity
All activities found.

physical_operational_intensity
All activities found.

information_digital_intensity
All activities found.

administrative_routine_intensity
All activities found.


In [40]:
feature_columns = [
    "cognitive_intensity",
    "social_intensity",
    "physical_operational_intensity",
    "information_digital_intensity",
    "administrative_routine_intensity"
]
display(
    features[feature_columns]
    .corr()
    .round(2)
)

,cognitive_intensity,social_intensity,physical_operational_intensity,information_digital_intensity,administrative_routine_intensity
cognitive_intensity,1.00,0.81,-0.11,0.89,0.89
social_intensity,0.81,1.00,-0.20,0.66,0.85
physical_operational_intensity,-0.11,-0.20,1.00,-0.12,-0.02
information_digital_intensity,0.89,0.66,-0.12,1.00,0.82
administrative_routine_intensity,0.89,0.85,-0.02,0.82,1.00


corr đã tạm ổn, insert data vào database

In [42]:

# ============================================================
# 1. Đổi tên các cột trong bảng occupation_features
# ============================================================

conn.execute("""
    ALTER TABLE occupation_features
    RENAME COLUMN physical_intensity
    TO physical_operational_intensity
""")

conn.execute("""
    ALTER TABLE occupation_features
    RENAME COLUMN digital_intensity
    TO information_digital_intensity
""")

conn.execute("""
    ALTER TABLE occupation_features
    RENAME COLUMN routine_intensity
    TO administrative_routine_intensity
""")

conn.commit()

print("Column names updated successfully.")

Column names updated successfully.


In [43]:
schema = pd.read_sql_query("""
    PRAGMA table_info(occupation_features)
""", conn)

display(schema)

,cid,name,type,notnull,dflt_value,pk
0,0,occupation_code,TEXT,0,None,1
1,1,cognitive_intensity,REAL,0,None,0
2,2,social_intensity,REAL,0,None,0
3,3,physical_operational_intensity,REAL,0,None,0
4,4,information_digital_intensity,REAL,0,None,0
5,5,administrative_routine_intensity,REAL,0,None,0


In [45]:
# ============================================================
# 3. Insert dữ liệu mới
# ============================================================

for _, row in features.iterrows():

    conn.execute("""
        INSERT INTO occupation_features (
            occupation_code,
            cognitive_intensity,
            social_intensity,
            physical_operational_intensity,
            information_digital_intensity,
            administrative_routine_intensity
        )
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row["occupation_code"],
        row["cognitive_intensity"],
        row["social_intensity"],
        row["physical_operational_intensity"],
        row["information_digital_intensity"],
        row["administrative_routine_intensity"]
    ))

conn.commit()

print("New occupation features inserted successfully.")

New occupation features inserted successfully.


In [46]:
check = pd.read_sql_query("""
    SELECT *
    FROM occupation_features
    LIMIT 10
""", conn)

display(check)

,occupation_code,cognitive_intensity,social_intensity,physical_operational_intensity,information_digital_intensity,administrative_routine_intensity
0,11-1011.00,4.88125,5.149333,1.524286,3.798,4.658333
1,11-1011.03,5.09500,4.902000,1.308571,4.058,4.275000
2,11-1021.00,3.93250,4.120000,2.061429,3.648,3.818333
3,11-1031.00,NaN,NaN,NaN,NaN,NaN
4,11-2011.00,4.16500,3.882000,1.381429,2.878,3.950000
5,11-2021.00,4.94750,4.749333,1.605714,4.168,4.078333
6,11-2022.00,4.53375,5.210000,0.714286,3.572,4.000000
7,11-2032.00,5.05500,5.142667,0.845714,3.796,4.465000
8,11-2033.00,4.06125,3.462667,1.110000,3.444,3.346667
9,11-3012.00,4.22000,3.959333,2.894286,3.762,4.363333


In [47]:
missing = pd.read_sql_query("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(cognitive_intensity IS NULL) AS missing_cognitive,
        SUM(social_intensity IS NULL) AS missing_social,
        SUM(physical_operational_intensity IS NULL) AS missing_physical,
        SUM(information_digital_intensity IS NULL) AS missing_digital,
        SUM(administrative_routine_intensity IS NULL) AS missing_routine
    FROM occupation_features
""", conn)

display(missing)

,total_rows,missing_cognitive,missing_social,missing_physical,missing_digital,missing_routine
0,1016,105,105,105,105,105


In [48]:
conn.close()